# Exploring HEST-1k Xenium data (TENX96, TENX99)

This notebook is a first look at two HEST-1k Xenium breast cancer samples downloaded
locally, ahead of the TERRA spatial + H&E multimodal project. Everything here uses
only the two samples already present at
`/lustre/scratch126/cellgen/lotfollahi/so25/hest_data` — nothing is downloaded.

We confirmed the on-disk layout first (`metadata/`, `st/`, `wsis/`, `cellvit_seg/`,
`xenium_seg/`, `patches/`, `tissue_seg/`, `transcripts/`) and it matches the standard
HEST-1k export, so we load everything through the HEST-Library's own `iter_hest`
loader rather than re-implementing any of the parsing.

In [ ]:
import os

import numpy as np
import pandas as pd
import geopandas as gpd
import h5py
import matplotlib.pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from shapely.geometry import box
from IPython.display import display

from hest import iter_hest

HEST_DIR = "/lustre/scratch126/cellgen/lotfollahi/so25/hest_data"
SAMPLE_IDS = ["TENX96", "TENX99"]

plt.rcParams["figure.dpi"] = 110

In [ ]:
# HEST-Library's own loader: reads st/, wsis/, metadata/, tissue_seg/ and
# cellvit_seg/ for each sample id and returns a HESTData object per sample.
samples = {st.meta["id"]: st for st in iter_hest(HEST_DIR, id_list=SAMPLE_IDS)}
list(samples.keys())

## 1. Sample inventory

Organ, disease, panel design and acquisition parameters straight from each
sample's `metadata/*.json` (written by HEST at curation time), plus the gene/cell
counts from the loaded `AnnData` and WSI objects.

Note: both samples are the **Xenium v1** targeted panel (280-gene predesigned
backbone + up to 100 custom add-on genes) — **neither is the newer Prime 5K**
panel. This matters for the "panel" ask below.

In [ ]:
def panel_label(meta):
    predesigned = meta.get("panel_num_targets_predesigned", 0)
    custom = meta.get("panel_num_targets_custom", 0)
    return f"Xenium v1 ({predesigned} predesigned + {custom} custom)"


rows = []
for sid, st in samples.items():
    m = st.meta
    rows.append(
        {
            "sample_id": sid,
            "organ": m["organ"],
            "disease": f'{m["disease_state"]} ({m["oncotree_code"]})',
            "panel": panel_label(m),
            "panel_name": m["panel_name"],
            "n_genes_panel": st.adata.n_vars,
            "n_cells_ST_matrix": st.adata.n_obs,
            "n_cells_xenium_segmentation": m["cells_under_tissue"],
            "wsi_width_px": m["fullres_px_width"],
            "wsi_height_px": m["fullres_px_height"],
            "pixel_size_um_per_px": m["pixel_size_um_estimated"],
            "scan_magnification": m["magnification"],
        }
    )

inventory = pd.DataFrame(rows).set_index("sample_id")
inventory

**First thing to flag:** `n_cells_ST_matrix` (rows in the shipped `.h5ad`) is
roughly **40-50x smaller** than `n_cells_xenium_segmentation` (the true
single-cell count reported in the metadata). The ST object is *not*
single-cell resolution — see Section 2 for why, and Section 5 for the true
single-cell counts from segmentation.

## 2. The spatial transcriptomics side

`st.adata` is a standard scanpy `AnnData`. Per the HEST-Library docs:
- `.obs` — one row per pooled ST unit, with `pxl_col_in_fullres`/`pxl_row_in_fullres`
  (pixel position on the full-res H&E), QC metrics (`total_counts`, `n_genes_by_counts`, ...).
- `.var` — one row per gene, with panel-level QC metrics.
- `.obsm['spatial']` — pixel coordinates on the full-resolution WSI.
- `.uns['spatial']` — a downscaled thumbnail + the scale factor to map fullres
  pixel coordinates onto it.

In [ ]:
for sid, st in samples.items():
    print(f"=== {sid} ===")
    print(st.adata)
    print()

### Wait — why so few "cells"?

HEST-1k pools Xenium single-cell data onto a **~100 micron pseudo-spot grid**,
matching Visium's spot pitch, so that Xenium and Visium samples can be
benchmarked on a common footing. Each row in `st.adata.obs` is therefore a
pooled bin containing on the order of dozens of real cells, not a single cell.
We confirm the ~100 um grid spacing directly below.

In [ ]:
# Confirm the ST object sits on a ~100 um grid rather than single-cell resolution
st96 = samples["TENX96"]
xs = np.sort(st96.adata.obs["pxl_col_in_fullres"].unique())
spacing_um = np.median(np.diff(xs)) * st96.pixel_size
print(
    f"Median pseudo-spot spacing: {spacing_um:.1f} um (Visium-like grid, not single-cell)"
)

for sid, st in samples.items():
    n_pooled = st.adata.n_obs
    n_true_cells = st.meta["cells_under_tissue"]
    print(
        f"{sid}: {n_pooled} pooled ST spots vs {n_true_cells} true Xenium cells "
        f"({n_true_cells / n_pooled:.1f}x more true cells than pooled spots)"
    )

In [ ]:
# Total transcript counts and genes detected, per pooled ST spot
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for col, (sid, st) in enumerate(samples.items()):
    axes[0, col].hist(st.adata.obs["total_counts"], bins=50, color="steelblue")
    axes[0, col].set_title(f"{sid}: transcript counts / pseudo-spot")
    axes[0, col].set_xlabel("total_counts")

    axes[1, col].hist(st.adata.obs["n_genes_by_counts"], bins=50, color="indianred")
    axes[1, col].set_title(f"{sid}: genes detected / pseudo-spot")
    axes[1, col].set_xlabel("n_genes_by_counts")
fig.tight_layout()

Here, `n_genes_by_counts` means how many spots there are (y-axis) with X many distinct genes detected (x-axis) and not their count.

In [ ]:
# Top 20 most-expressed genes per sample
for sid, st in samples.items():
    print(f"=== {sid}: top 20 genes by total counts ===")
    top20 = st.adata.var.sort_values("total_counts", ascending=False).head(20)
    display(top20[["total_counts", "mean_counts", "n_cells_by_counts"]])

In [ ]:
# Empty spots / outliers
for sid, st in samples.items():
    obs = st.adata.obs
    n_empty = int((obs["total_counts"] == 0).sum())
    print(
        f"{sid}: {n_empty} empty pseudo-spots (0 counts), "
        f"max total_counts={obs['total_counts'].max():.0f}, "
        f"99th pct total_counts={obs['total_counts'].quantile(0.99):.0f}"
    )

### Gene panel differs between the two samples

Both samples report `n_genes_panel = 541`, but that doesn't mean they use the
*same* 541 genes.

In [ ]:
g96 = set(samples["TENX96"].adata.var_names)
g99 = set(samples["TENX99"].adata.var_names)
print(f"Shared genes: {len(g96 & g99)}")
print(f"Unique to TENX96: {len(g96 - g99)}")
print(f"Unique to TENX99: {len(g99 - g96)}")

## 3. The H&E side

The full-resolution WSI is exposed as `st.wsi` (a `trident` WSI object wrapped
by HEST), lazily backed by the pyramidal `.tif`. We report the pyramid levels,
show a thumbnail, and display a handful of the 224x224 px / 0.5 um/px (20x)
patches HEST pre-extracted into `patches/*.h5`.

Note: the metadata's `magnification: 40x` refers to the *scan* magnification
(native pixel size 0.2125 um/px). The 224x224 patches requested at "20x"
(0.5 um/px) are a 2x downsample of that native resolution — HEST already
extracted them this way (`patch_size_src=527` native px -> `224` px target,
`527 * 0.2125 um = 224 * 0.5 um = 112 um`, confirming the patches are exactly
20x-equivalent).

In [ ]:
rows = []
for sid, st in samples.items():
    wsi = st.wsi
    for lvl, (dim, ds) in enumerate(zip(wsi.level_dimensions, wsi.level_downsamples)):
        rows.append(
            {
                "sample_id": sid,
                "level": lvl,
                "width": dim[0],
                "height": dim[1],
                "downsample": round(ds, 2),
            }
        )
pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (sid, st) in zip(axes, samples.items()):
    thumb = st.wsi.get_thumbnail((600, 600))
    ax.imshow(thumb)
    ax.set_title(f"{sid}: thumbnail")
    ax.axis("off")
fig.tight_layout()

In [ ]:
# 4 random pre-extracted 224x224 px patches (0.5 um/px, i.e. 20x) per sample
rng = np.random.default_rng(0)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, sid in enumerate(SAMPLE_IDS):
    with h5py.File(os.path.join(HEST_DIR, "patches", f"{sid}.h5"), "r") as f:
        idx = np.sort(rng.choice(f["img"].shape[0], size=4, replace=False))
        for col, i in enumerate(idx):
            axes[row, col].imshow(f["img"][i])
            axes[row, col].set_title(f"{sid} patch #{i}")
            axes[row, col].axis("off")
fig.suptitle("Random 224x224 px patches at 20x (0.5 um/px)")
fig.tight_layout()

## 4. Alignment sanity check

This is the important part: do the ST pixel coordinates actually line up with
the H&E pixels? First we overlay every pooled ST spot centroid on the
downscaled thumbnail. Then we zoom into a genuinely dense ~500x500 um region
at full resolution and overlay individual Xenium cell centroids and CellViT
nuclear boundaries directly on the H&E pixels.

We use the true single-cell segmentation (`xenium_seg/`, `cellvit_seg/`) for
the zoomed-in check rather than the pooled ST spots, since those are the
actual per-cell coordinates we need to validate.

In [ ]:
# Load the true single-cell segmentation (already in full-res H&E pixel space)
xenium_cells = {}
cellvit_cells = {}
for sid in SAMPLE_IDS:
    xc = gpd.read_parquet(
        os.path.join(HEST_DIR, "xenium_seg", f"{sid}_xenium_cell_seg.parquet")
    )
    xc["cx"] = xc.geometry.centroid.x
    xc["cy"] = xc.geometry.centroid.y
    xenium_cells[sid] = xc

    cv = gpd.read_parquet(
        os.path.join(HEST_DIR, "cellvit_seg", f"{sid}_cellvit_seg.parquet")
    )
    cv["cx"] = cv.geometry.centroid.x
    cv["cy"] = cv.geometry.centroid.y
    cellvit_cells[sid] = cv

{sid: (len(xenium_cells[sid]), len(cellvit_cells[sid])) for sid in SAMPLE_IDS}

In [ ]:
# Step 1: overlay ST pseudo-spot centroids on the downscaled thumbnail
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (sid, st) in zip(axes, samples.items()):
    scalef = st.adata.uns["spatial"]["ST"]["scalefactors"][
        "tissue_downscaled_fullres_scalef"
    ]
    thumb = st.adata.uns["spatial"]["ST"]["images"]["downscaled_fullres"]
    xy = st.adata.obsm["spatial"] * scalef

    ax.imshow(thumb)
    ax.scatter(xy[:, 0], xy[:, 1], s=3, c="red", alpha=0.6)
    ax.set_title(f"{sid}: ST spot centroids on H&E thumbnail")
    ax.axis("off")
fig.tight_layout()

In [ ]:
# Find a genuinely dense region to zoom into, using true single-cell centroids
def find_dense_region_center(cells_gdf, bin_size_um=50, pixel_size=0.2125):
    bin_px = bin_size_um / pixel_size
    xb = (cells_gdf["cx"] // bin_px).astype(int)
    yb = (cells_gdf["cy"] // bin_px).astype(int)
    top_bin = pd.DataFrame({"xb": xb, "yb": yb}).value_counts().index[0]
    xb0, yb0 = top_bin
    return (xb0 + 0.5) * bin_px, (yb0 + 0.5) * bin_px


dense_centers = {
    sid: find_dense_region_center(xenium_cells[sid], pixel_size=samples[sid].pixel_size)
    for sid in SAMPLE_IDS
}
dense_centers

In [ ]:
def get_crop(st, center_xy, size_um):
    px = st.pixel_size
    size_px = int(round(size_um / px))
    cx, cy = center_xy
    x0 = int(cx - size_px / 2)
    y0 = int(cy - size_px / 2)
    region = st.wsi.read_region(
        (x0, y0), level=0, size=(size_px, size_px), read_as="numpy"
    )
    return region, x0, y0, size_px

In [ ]:
# Step 2: zoom to a 500x500 um crop at full resolution, overlay true cell
# centroids and CellViT nuclear boundaries directly on the H&E pixels
fig, axes = plt.subplots(2, 2, figsize=(13, 13))
for col, sid in enumerate(SAMPLE_IDS):
    st = samples[sid]
    region, x0, y0, size_px = get_crop(st, dense_centers[sid], size_um=500)

    axes[0, col].imshow(region)
    axes[0, col].set_title(f"{sid}: H&E only (500x500 um)")
    axes[0, col].axis("off")

    axes[1, col].imshow(region)

    xc = xenium_cells[sid]
    in_view = xc["cx"].between(x0, x0 + size_px) & xc["cy"].between(y0, y0 + size_px)
    axes[1, col].scatter(
        xc.loc[in_view, "cx"] - x0,
        xc.loc[in_view, "cy"] - y0,
        s=8,
        c="lime",
        label="Xenium cell centroid",
    )

    cv = cellvit_cells[sid]
    cv_in_view = cv["cx"].between(x0, x0 + size_px) & cv["cy"].between(y0, y0 + size_px)
    for geom in cv.loc[cv_in_view, "geometry"]:
        ex, ey = geom.exterior.xy
        axes[1, col].plot(np.array(ex) - x0, np.array(ey) - y0, c="cyan", linewidth=0.6)

    axes[1, col].set_title(f"{sid}: + Xenium centroids (green) & CellViT nuclei (cyan)")
    axes[1, col].axis("off")
    axes[1, col].legend(loc="lower right", fontsize=8)
    axes[1, col].add_artist(ScaleBar(st.pixel_size, "um", location="lower left"))
fig.tight_layout()

Look closely at the bottom row: the green centroids and cyan nuclear outlines
should sit exactly on top of visible nuclei in the H&E crop. If they were
offset or rotated relative to the tissue, that would indicate a registration
problem between the ST/segmentation coordinate system and the WSI.

## 5. Cell segmentation

Two independent segmentations ship with each sample:
- **CellViT** — a deep-learning nuclear segmenter run directly on the H&E image.
- **Xenium native** — DAPI-based cell/nucleus calls from the Xenium instrument,
  non-rigidly aligned to H&E by HEST.

We visualize CellViT boundaries on a small H&E crop, then compare *three*
different "cell count" numbers per sample (they measure different things).

In [ ]:
# CellViT nuclear boundaries on a small (100x100 um) H&E crop, colored by class
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
for ax, sid in zip(axes, SAMPLE_IDS):
    st = samples[sid]
    region, x0, y0, size_px = get_crop(st, dense_centers[sid], size_um=100)
    ax.imshow(region)

    cv = cellvit_cells[sid]
    in_view = cv["cx"].between(x0, x0 + size_px) & cv["cy"].between(y0, y0 + size_px)
    for _, row in cv.loc[in_view].iterrows():
        ex, ey = row.geometry.exterior.xy
        color = "yellow" if row["class"] == "Neoplastic" else "cyan"
        ax.plot(np.array(ex) - x0, np.array(ey) - y0, c=color, linewidth=1)

    ax.set_title(f"{sid}: CellViT nuclei (yellow=Neoplastic, cyan=other)")
    ax.axis("off")
    ax.add_artist(ScaleBar(st.pixel_size, "um", location="lower left"))
fig.tight_layout()

In [ ]:
# Three different "cell count" numbers, and why they differ
rows = []
for sid, st in samples.items():
    n_cellvit = len(cellvit_cells[sid])
    n_xenium = len(xenium_cells[sid])
    n_st = st.adata.n_obs
    rows.append(
        {
            "sample_id": sid,
            "cellvit_nuclei_on_HE": n_cellvit,
            "xenium_native_cells": n_xenium,
            "ST_matrix_pseudo_spots": n_st,
            "cellvit_vs_xenium_native_pct_diff": round(
                100 * (n_cellvit - n_xenium) / n_xenium, 1
            ),
        }
    )
pd.DataFrame(rows).set_index("sample_id")

In [ ]:
# CellViT cell-type composition, for context
for sid in SAMPLE_IDS:
    print(f"=== {sid}: CellViT class composition ===")
    print(cellvit_cells[sid]["class"].value_counts())
    print()

`cellvit_nuclei_on_HE` and `xenium_native_cells` are two independent estimates
of the *same* underlying single-cell count — one from an image-only nuclear
detector, one from the Xenium instrument's own DAPI-based caller aligned onto
the H&E. `ST_matrix_pseudo_spots` is unrelated: it's the count of pooled
100 um bins in the shipped `AnnData`, ~40-50x smaller (Section 2).

## 6. Spatial structure

Do we see visible spatial patterns in the raw data? We plot at the same
pooled ST spot resolution used in Section 2 (not the true single-cell
segmentation), because gene expression values only exist at that resolution
in the shipped `AnnData` — recovering true single-cell expression would
require aggregating the raw per-transcript files (53M-127M rows each), which
isn't needed to answer this question.

Markers: **EPCAM** (epithelial/tumor), **PTPRC** (immune, CD45), **PECAM1**
(endothelial, CD31) — all three are present in both samples' panels despite
the panel differences noted in Section 2.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
for ax, (sid, st) in zip(axes, samples.items()):
    xy = st.adata.obsm["spatial"]
    sca = ax.scatter(
        xy[:, 0], xy[:, 1], c=st.adata.obs["total_counts"], s=5, cmap="viridis"
    )
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.set_title(f"{sid}: total transcript counts per pseudo-spot")
    ax.axis("off")
    fig.colorbar(sca, ax=ax, shrink=0.7)
fig.tight_layout()

In [ ]:
markers = ["EPCAM", "PTPRC", "PECAM1"]
fig, axes = plt.subplots(
    len(SAMPLE_IDS), len(markers), figsize=(4 * len(markers), 4 * len(SAMPLE_IDS))
)
for row, (sid, st) in enumerate(samples.items()):
    xy = st.adata.obsm["spatial"]
    for col, gene in enumerate(markers):
        expr = np.asarray(st.adata[:, gene].X).flatten()
        ax = axes[row, col]
        sca = ax.scatter(xy[:, 0], xy[:, 1], c=expr, s=5, cmap="magma")
        ax.invert_yaxis()
        ax.set_aspect("equal")
        ax.set_title(f"{sid}: {gene}")
        ax.axis("off")
fig.tight_layout()

## 7. Patch-level cell density

We tile each WSI into a fresh, non-overlapping grid of 224x224 px patches at
20x (0.5 um/px — i.e. 527x527 px in the native 0.2125 um/px image), keep only
patches that overlap the tissue mask (`st.tissue_contours`), and count true
single-cell centroids (`xenium_seg`) landing in each patch.

Note this is a *different* patch grid from the pre-extracted `patches/*.h5`
used in Section 3: those are centered on (and overlap between) the pooled
100 um ST spots, so they wouldn't give an unbiased census of cell density
across the whole tissue area. This section builds a proper non-overlapping
tiling instead.

In [ ]:
def tile_wsi(st, tissue_gdf, target_patch_px=224, target_pixel_size_um=0.5):
    native_stride = int(round(target_patch_px * target_pixel_size_um / st.pixel_size))
    width, height = st.wsi.dimensions
    xs = np.arange(0, width, native_stride)
    ys = np.arange(0, height, native_stride)
    boxes = [box(x, y, x + native_stride, y + native_stride) for y in ys for x in xs]
    patches = gpd.GeoDataFrame({"geometry": boxes})

    tissue_union = tissue_gdf.union_all()
    patches = patches[patches.intersects(tissue_union)].reset_index(drop=True)
    return patches, native_stride

In [ ]:
density_results = {}
for sid, st in samples.items():
    patches, stride = tile_wsi(st, st.tissue_contours)

    centroids = xenium_cells[sid].copy()
    centroids["geometry"] = gpd.points_from_xy(centroids["cx"], centroids["cy"])

    joined = gpd.sjoin(centroids, patches, predicate="within", how="inner")
    counts = joined.groupby("index_right").size().reindex(patches.index, fill_value=0)
    density_results[sid] = counts

    print(
        f"{sid}: {len(patches)} tissue-overlapping patches (stride={stride} native px), "
        f"median={counts.median():.0f}, p95={counts.quantile(0.95):.0f}, max={counts.max()}"
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, sid in zip(axes, SAMPLE_IDS):
    counts = density_results[sid]
    ax.hist(counts, bins=50, color="teal")
    ax.axvline(counts.median(), color="black", linestyle="--", label="median")
    ax.axvline(counts.quantile(0.95), color="red", linestyle="--", label="95th pct")
    ax.set_title(f"{sid}: cells per 224x224 px (20x) patch")
    ax.set_xlabel("cell count")
    ax.legend()
fig.tight_layout()

In [ ]:
pd.DataFrame(
    {
        sid: {
            "n_tissue_patches": len(density_results[sid]),
            "median_cells_per_patch": density_results[sid].median(),
            "p95_cells_per_patch": density_results[sid].quantile(0.95),
            "max_cells_per_patch": density_results[sid].max(),
        }
        for sid in SAMPLE_IDS
    }
).T

## 8. Summary observations

**1. The shipped `.h5ad` is not single-cell resolution.** HEST pools Xenium
calls onto a ~100 um pseudo-spot grid (confirmed: median spot spacing exactly
100 um). TENX96 has 7,233 pooled spots vs 365,604 true cells (50.5x fewer);
TENX99 has 23,060 pooled spots vs 892,966 true cells (38.7x fewer). Anyone
using the `.h5ad` directly for cell-level modelling needs to know it is
spot-level, not cell-level.

**2. The two panels are not the same 541 genes.** Both samples report
`n_genes_panel = 541`, but only 441 are shared — 100 differ each way.
TENX96's unique genes are enriched for luminal/epithelial markers (its top-20
by counts includes MUC1, GATA3, ESR1, FOXA1, ERBB2), consistent with its
ILC (lobular carcinoma) annotation. Oddly, TENX99's metadata claims
`panel_num_targets_custom = 0` (no add-on), yet it still shares only 441/541
genes with TENX96 — the metadata's custom-target count doesn't fully explain
the observed panel difference and would be worth double-checking against 10x's
official panel spec if exact gene identity matters downstream.

**3. TENX99 is very stroma-dominated.** Its top expressed genes are ECM/
stromal (LUM, POSTN, MMP2, ACTA2, DSP) rather than epithelial, and this
matches what's visible directly in the H&E crops (Sections 4-5): the TENX99
region sampled is sparse, fibrous stroma with few, spindle-shaped nuclei,
versus TENX96's crop which is cellular (a dense immune aggregate plus
adipose tissue).

**4. Alignment is good.** Visually confirmed at both scales: the pooled ST
spot grid follows the tissue silhouette on the thumbnail for both samples,
and at full resolution the Xenium cell centroids and CellViT nuclear outlines
sit precisely on visible nuclei in H&E — including landing exactly on a dense
lymphocyte aggregate in the TENX96 crop. No offset or rotation was seen in
either sample.

**5. CellViT under-calls nuclei relative to Xenium's native segmentation,
more so in stroma-rich tissue.** CellViT finds 12.1% fewer nuclei than the
Xenium DAPI-based caller in TENX96 (321,316 vs 365,604) and 20.4% fewer in
TENX99 (710,379 vs 892,966). The full-resolution crop suggests why: in
TENX99's fibrous stroma, several visibly spindle-shaped nuclei in the H&E
were not outlined by CellViT, consistent with a larger miss rate in that
tissue type. CellViT class composition also differs a lot: TENX96 is ~84%
"Connective" / 9.5% "Neoplastic", TENX99 is ~63% "Connective" / 34%
"Neoplastic" — broadly consistent with the two different carcinoma subtypes,
though this is a rough consistency check, not a validation.

**6. Patch-level cell density (224x224 px @ 20x, tissue-only, true
single-cell counts):** TENX96 — 5,023 tissue patches, median 73 cells/patch,
95th pct 151, max 276. TENX99 — 16,511 tissue patches, median 53, 95th pct
103, max 190. For sizing TERRA's per-patch cell cap: a budget around
150-280 would cover the 95th percentile on both samples; if a single fixed
cap needs to be shared across future, possibly denser, samples, rounding up
toward ~300 would avoid truncation on data like this.

**7. No corrupted or empty spots found.** Zero pseudo-spots with 0 total
counts in either sample, and no other obviously broken values in the QC
histograms.